### Restart and Run All

In [2]:
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

year = 2025
quarter = 2
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:06:08 14:52:23


In [3]:
cols = 'name year quarter q_amt_c q_amt_p inc_profit percent'.split()

format_dict = {
                'q_amt':'{:,}','q_amt_c':'{:,}','q_amt_p':'{:,}','inc_profit':'{:,}',
                'yoy_gain':'{:,}','acc_gain':'{:,}',    
                'q_eps':'{:.4f}','y_eps':'{:.4f}','aq_eps':'{:.4f}','ay_eps':'{:.4f}',
                'percent':'{:.2f}%'
              }

In [4]:
sql = '''
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s AND quarter <= %s) 
OR (year = %s-1 AND quarter >= %s+1)
ORDER BY year DESC, quarter DESC'''
sql = sql % (year,quarter,year,quarter)
print(sql)


SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = 2025 AND quarter <= 2) 
OR (year = 2025-1 AND quarter >= 2+1)
ORDER BY year DESC, quarter DESC


In [5]:
dfc = pd.read_sql(sql, conlt)
dfc['Counter'] = 1
dfc_grp = dfc.groupby(['name'], as_index=False).sum()
dfc_grp = dfc_grp[dfc_grp['Counter'] == 4]
dfc_grp.shape

(196, 5)

In [6]:
sql = '''
SELECT name,year,quarter,q_amt
FROM epss 
WHERE (year = %s-1 AND quarter <= %s - 1) 
OR (year = %s-2 AND quarter >= %s)
ORDER BY year DESC, quarter DESC'''
sql = sql % (year,quarter,year,quarter)
dfp = pd.read_sql(sql, conlt)
dfp['Counter'] = 1
dfp_grp = dfp.groupby(['name'], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp['Counter'] == 4]
dfp_grp.shape

(202, 5)

In [7]:
dfp = pd.read_sql(sql, conlt)
dfp["Counter"] = 1
dfp_grp = dfp.groupby(["name"], as_index=False).sum()
dfp_grp = dfp_grp[dfp_grp["Counter"] == 4]
dfp_grp.head().style.format(format_dict)

,name,year,quarter,q_amt,Counter
0,3BBIF,8093,10,"-8,619,852",4
1,ACE,8093,10,"1,052,706",4
2,ADVANC,8093,10,"30,780,229",4
3,AEONTS,8093,10,"3,165,741",4
4,AH,8093,10,"1,368,109",4


In [8]:
dfm = pd.merge(dfc_grp, dfp_grp, on="name", suffixes=(["_c", "_p"]), how="inner")
dfm["inc_profit"] = dfm["q_amt_c"] - dfm["q_amt_p"]
dfm["percent"] = round(dfm["inc_profit"] / abs(dfm["q_amt_p"]) * 100, 2)
dfm["year"] = year
dfm["quarter"] = "Q" + str(quarter)
df_percent = dfm[cols]
df_percent.head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q2,"8,709,489","-8,619,852","17,329,341",201.04%
1,ACE,2025,Q2,"738,812","1,052,706","-313,894",-29.82%
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
3,AEONTS,2025,Q2,"3,077,419","3,165,741","-88,322",-2.79%
4,AH,2025,Q2,"738,988","1,368,109","-629,121",-45.98%


In [9]:
# Create the SQL query with parameter binding
sql = text("DELETE FROM yr_profits WHERE year = :year AND quarter = :quarter")

# Execute the query with parameters
params = {'year': year, 'quarter': f'Q{quarter}'}
rp = conlt.execute(sql, params)

# Print the number of rows affected
print("Rows deleted:", rp.rowcount)

Rows deleted: 202


In [10]:
sql = "SELECT name, id FROM tickers"
tickers = pd.read_sql(sql, conlt)
df_ins = pd.merge(df_percent, tickers, on="name", how="inner")
rcds = df_ins.values.tolist()
len(rcds)

195

In [11]:
# Convert DataFrame to list of records
rcds = df_ins.values.tolist()

# Define column names in the same order as values
columns = ['name', 'year', 'quarter', 'latest_amt', 'previous_amt', 'inc_amt', 'inc_pct', 'ticker_id']

# SQL insert statement with named parameters
sql = text("""
    INSERT INTO yr_profits 
    (name, year, quarter, latest_amt, previous_amt, inc_amt, inc_pct, ticker_id)
    VALUES (:name, :year, :quarter, :latest_amt, :previous_amt, :inc_amt, :inc_pct, :ticker_id)
""")

try:
    # Execute inserts
    for rcd in rcds:
        # Convert list to dictionary
        params = dict(zip(columns, rcd))
        conlt.execute(sql, params)
except Exception as e:
    raise e

### End of loop

In [13]:
criteria_1 = df_ins.q_amt_c > 440_000
df_ins.loc[criteria_1, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q2,"8,709,489","-8,619,852","17,329,341",201.04%
1,ACE,2025,Q2,"738,812","1,052,706","-313,894",-29.82%
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
3,AEONTS,2025,Q2,"3,077,419","3,165,741","-88,322",-2.79%
4,AH,2025,Q2,"738,988","1,368,109","-629,121",-45.98%


In [14]:
criteria_2 = df_ins.q_amt_p > 400_000
df_ins.loc[criteria_2, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
1,ACE,2025,Q2,"738,812","1,052,706","-313,894",-29.82%
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
3,AEONTS,2025,Q2,"3,077,419","3,165,741","-88,322",-2.79%
4,AH,2025,Q2,"738,988","1,368,109","-629,121",-45.98%
6,AIMIRT,2025,Q2,"827,111","485,997","341,114",70.19%


In [15]:
criteria_3 = df_ins.percent > 10.00
df_ins.loc[criteria_3, cols].head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
0,3BBIF,2025,Q2,"8,709,489","-8,619,852","17,329,341",201.04%
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
5,AIE,2025,Q2,"148,430","81,612","66,818",81.87%
6,AIMIRT,2025,Q2,"827,111","485,997","341,114",70.19%
8,AMATA,2025,Q2,"2,757,200","1,856,358","900,842",48.53%


In [16]:
final_criteria = criteria_1 & criteria_2 & criteria_3
df_ins.loc[final_criteria, cols].sort_values(by=["percent"], ascending=[False]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
128,RCL,2025,Q2,"11,472,607","1,201,533","10,271,074",854.83%
67,GULF,2025,Q2,"95,184,877","14,506,279","80,678,598",556.16%
183,TVO,2025,Q2,"2,425,616","909,838","1,515,778",166.60%
130,ROJNA,2025,Q2,"2,220,836","877,057","1,343,779",153.21%
43,CKP,2025,Q2,"2,412,207","1,105,431","1,306,776",118.21%


In [17]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
6,AIMIRT,2025,Q2,"827,111","485,997","341,114",70.19%
8,AMATA,2025,Q2,"2,757,200","1,856,358","900,842",48.53%
10,AOT,2025,Q2,"19,232,337","13,011,133","6,221,204",47.81%
12,ASIAN,2025,Q2,"713,985","497,519","216,466",43.51%


In [18]:
df_ins.loc[final_criteria, cols].sort_values(by=["name"], ascending=[True]).head().style.format(format_dict)

,name,year,quarter,q_amt_c,q_amt_p,inc_profit,percent
2,ADVANC,2025,Q2,"39,612,451","30,780,229","8,832,222",28.69%
6,AIMIRT,2025,Q2,"827,111","485,997","341,114",70.19%
8,AMATA,2025,Q2,"2,757,200","1,856,358","900,842",48.53%
10,AOT,2025,Q2,"19,232,337","13,011,133","6,221,204",47.81%
12,ASIAN,2025,Q2,"713,985","497,519","216,466",43.51%


In [19]:
conlt.commit()
conlt.close()

In [20]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:06:08 14:52:24
